# Demand Predictor — Train on Colab (GPU)

Trains the `DemandPredictor` U-Net (Stage 3 of the completing-router overhaul). It learns to
predict, per layer, **where the next K nets will route**, so the current net can be routed to leave
space for them. Supervised — labels come from boards completed by the rip-up-and-reroute router.

**Before you run:** make sure these files are pushed to GitHub (they are new):
`pcb_router/routing/rip_up_router.py`, `pcb_router/models/demand_predictor.py`,
`scripts/train_demand_predictor.py`.

Everything (dataset, checkpoints, training-progress images) is saved to **Google Drive** so it
survives disconnects/resets — see the Drive-mount cell below.

**Runtime:** set Runtime → Change runtime type → GPU.

## 0. Setup

In [ ]:
import os, sys
REPO = '/content/Router'
if not os.path.exists(REPO):
    !git clone https://github.com/Klutzhehe/Router.git $REPO
os.chdir(REPO)
!git pull --ff-only
if REPO not in sys.path:
    sys.path.insert(0, REPO)   # chdir alone does NOT put the repo on sys.path; imports need this
!pip -q install scipy pyyaml
import torch
assert os.path.exists('scripts/train_demand_predictor.py'), 'new files missing - did git pull fetch them?'
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), '| files OK')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Everything (dataset, checkpoints, training-progress images) is saved under this Drive folder so
# it survives Colab disconnects/resets. Change the folder name if you want a different project.
DRIVE_DIR = '/content/drive/MyDrive/pcb_router'
os.makedirs(f'{DRIVE_DIR}/data', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints/viz', exist_ok=True)

DATA_PATH = f'{DRIVE_DIR}/data/demand_dataset.pkl'
CKPT_PATH = f'{DRIVE_DIR}/checkpoints/demand_predictor.pt'
VIZ_DIR   = f'{DRIVE_DIR}/checkpoints/viz'
print('Saving dataset/checkpoints/viz to Google Drive at:', DRIVE_DIR)

## 1. Generate the dataset

Routes boards with the rip-up-and-reroute router and keeps only fully-routed boards (clean labels).
Use **multi-layer** stages — they complete far more reliably than single-layer congested ones
(single-layer boards can have pins sealed by other nets' pads). Start small to gauge speed, then
raise `boards_per_stage`. Saved straight to Google Drive.

In [ ]:
from scripts.train_demand_predictor import generate_dataset

samples = generate_dataset(
    stage_names=['s10_via_plus_multi_net'],   # multi-layer; add e.g. 's11_...' for variety
    boards_per_stage=60,
    out_path=DATA_PATH,   # -> Google Drive (set in the Drive-mount cell above)
    max_iters=8,
)
print('boards generated:', len(samples))

## 2. Train the predictor

Weighted BCE (route cells are ~0.1% of pixels, so positives are up-weighted). Batches pad to the
batch's max size, so no wasted compute on small boards. Streams one line per epoch to the log
(loss, on-route vs. off-route predicted probability, elapsed time). Checkpoints to **Google Drive**
every `SAVE_EVERY` epochs (and at the end), and can **resume** from there if the runtime disconnects.

In [ ]:
import os, time, pickle, torch
from torch.utils.data import DataLoader
from scripts.train_demand_predictor import DemandDataset, collate_pad, demand_loss
from pcb_router.models.demand_predictor import DemandPredictor

# ---- config (edit me) ----
DATA       = DATA_PATH     # Google Drive path, set in the Drive-mount cell above
EPOCHS     = 80
BATCH      = 8
LR         = 2e-3
K          = 3
CKPT       = CKPT_PATH     # Google Drive path -> checkpoint survives disconnects
SAVE_EVERY = 10            # checkpoint to Drive every N epochs, plus always at the end
RESUME     = True          # continue from CKPT if it already exists in Drive
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---- data + model ----
samples = pickle.load(open(DATA, 'rb'))
ds = DemandDataset(samples, K=K)
dl = DataLoader(ds, batch_size=BATCH, shuffle=True, collate_fn=collate_pad)
model = DemandPredictor().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

start_epoch = 0
if RESUME and os.path.exists(CKPT):
    ckpt = torch.load(CKPT, map_location=device)
    model.load_state_dict(ckpt['model'])
    if 'opt' in ckpt:
        opt.load_state_dict(ckpt['opt'])
    start_epoch = ckpt.get('epoch', 0)
    print(f'resumed from {CKPT} at epoch {start_epoch}', flush=True)

print(f'{len(ds)} samples from {len(samples)} boards | device={device}', flush=True)

# Fixed eval batch for a quick quality metric each epoch (predicted prob ON vs OFF the true routes).
n_eval = min(16, len(ds))
exb, eyb, elmb, esmb = collate_pad([ds[i] for i in range(n_eval)])
exb, eyb, elmb, esmb = exb.to(device), eyb.to(device), elmb.to(device), esmb.to(device)

def save_checkpoint(epoch):
    os.makedirs(os.path.dirname(CKPT) or '.', exist_ok=True)
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'K': K, 'epoch': epoch}, CKPT)

# ---- training loop (streams one line per epoch to the log; checkpoints to Drive) ----
t0 = time.time()
for ep in range(start_epoch, EPOCHS):
    model.train(); tot = nb = 0
    for x, y, lm, sm in dl:
        x, y, lm, sm = x.to(device), y.to(device), lm.to(device), sm.to(device)
        loss = demand_loss(model(x), y, lm, sm)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        tot += loss.item(); nb += 1
    model.eval()
    with torch.no_grad():
        p = model.predict(exb, elmb)
    on  = p[eyb > 0.5].mean().item()                                  # want this to climb
    off = p[(eyb < 0.5) & (esmb.expand_as(eyb) > 0)].mean().item()    # want this low
    print(f'epoch {ep+1:3d}/{EPOCHS} | loss {tot/nb:.4f} | on-route {on:.3f}  off-route {off:.3f}  '
          f'(gap {on-off:+.3f}) | {time.time()-t0:5.0f}s', flush=True)

    if (ep + 1) % SAVE_EVERY == 0 or ep == EPOCHS - 1:
        save_checkpoint(ep + 1)
        print(f'  checkpoint saved -> {CKPT}', flush=True)

print('done. final checkpoint at', CKPT, flush=True)

## 3. Sanity check — predicted demand vs. actual future routes

On a trained model the **PREDICTED demand** should light up where the **ACTUAL future routes** are
(and along the corridor between the next-K pins). Also saves the figure to Google Drive.

In [ ]:
import pickle
import matplotlib.pyplot as plt
import torch
from pcb_router.models.demand_predictor import DemandPredictor
from scripts.train_demand_predictor import DemandDataset, collate_pad

samples = pickle.load(open(DATA_PATH, 'rb'))
ds = DemandDataset(samples, K=3)

model = DemandPredictor().cuda()
model.load_state_dict(torch.load(CKPT_PATH, map_location='cuda')['model'])
model.eval()

i = 0                        # try different sample indices
x, y, lm = ds[i]
xb, yb, lmb, smb = collate_pad([ds[i]])
with torch.no_grad():
    p = model.predict(xb.cuda(), lmb.cuda())[0].cpu()

L0 = 0
fig, ax = plt.subplots(1, 4, figsize=(20, 5))
ax[0].imshow(x[8]);   ax[0].set_title('current net pins')
ax[1].imshow(x[9]);   ax[1].set_title('next-K nets pins')
ax[2].imshow(p[L0]);  ax[2].set_title('PREDICTED demand (L0)')
ax[3].imshow(y[L0]);  ax[3].set_title('ACTUAL future routes (L0)')
for a in ax:
    a.axis('off')
plt.tight_layout()
fig.savefig(f'{VIZ_DIR}/sanity_check.png', dpi=100, bbox_inches='tight')  # -> Google Drive
plt.show()